#**GHI CHÚ:**
 + Các mục lớn 1,2,3... là các mục lớn, bên trong nó sẽ còn có chi nhỏ (người được phụ trách tự chia), ví dụ 4.1, 4.2, ... sau đó là các cell code.
 + Sau khi train phải lưu:
    - Accuracy
    - Precision
    - Recall
    - F1-score
    - Confusion Matrix
    - Classification Report

Đặc biệt ưu tiên Macro F1 đối với dataset mất cân bằng.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#1. Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q emoji

#2. Kiểm tra thư viện

In [ ]:
import transformers
import datasets
import sklearn
import pandas
import numpy

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)

Transformers: 5.20.2
Datasets: 4.8.5
Scikit-learn: 1.6.1
Pandas: 2.2.3
NumPy: 2.1.3


#3. Kiểm tra PhoBERT

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = "vinai/phobert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

print("PhoBERT loaded successfully!")

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PhoBERT loaded successfully!


#4. Tiền xủ lý dữ liệu (tiền xử lý, tokenizer, chuyển nhãn) - Hưng, Hoàng Minh, Minh Tú


# 4.1 Làm sạch Text

In [ ]:
import re
import pandas as pd
import emoji
from datasets import load_dataset, concatenate_datasets
from huggingface_hub import login

# Đăng nhập để tải được dataset có khóa uitnlp/vihsd
# login("")

print("Đang tải các dataset trực tiếp từ Hugging Face...")

# 1. Tải dataset Toxicity: tarudesu/ViCTSD
# Dataset có cột 'Comment' chứa văn bản
ds_toxic = load_dataset("tarudesu/ViCTSD")
df_toxic = concatenate_datasets([ds_toxic['train'], ds_toxic['test'], ds_toxic['validation']]).to_pandas()
df_toxic = df_toxic[['Comment', 'Toxicity']].rename(
    columns={'Comment': 'text', 'Toxicity': 'label_raw'}
)
df_toxic['task'] = 'toxicity'

# 2. Tải dataset Hate Speech: uitnlp/vihsd
# Dataset có cột 'free_text' chứa văn bản
ds_hate = load_dataset("uitnlp/vihsd")
df_hate = concatenate_datasets([ds_hate['train'], ds_hate['test'], ds_hate['validation']]).to_pandas()
df_hate = df_hate[['free_text', 'label_id']].rename(
    columns={'free_text': 'text', 'label_id': 'label_raw'}
)
df_hate['task'] = 'hate_speech'

# 3. Tải dataset Sentiment: ura-hcmut/UIT-VSFC
# Dataset có cột chứa văn bản và cột 'sentiment'/'label'
ds_sentiment = load_dataset("ura-hcmut/UIT-VSFC")
df_sent = concatenate_datasets([ds_sentiment['train'], ds_sentiment['test'], ds_sentiment['validation']]).to_pandas()

# Tự động lấy đúng cột văn bản (tránh lỗi KeyError 'sentence')
text_col = [col for col in ['sentence', 'Sentence', 'text', 'Comment'] if col in df_sent.columns][0]
label_col = 'sentiment' if 'sentiment' in df_sent.columns else 'label'

df_sent = df_sent[[text_col, label_col]].rename(
    columns={text_col: 'text', label_col: 'label_raw'}
)
df_sent['task'] = 'sentiment'

# Gộp tất cả văn bản vào một DataFrame duy nhất để xử lý làm sạch tập trung
df = pd.concat([df_toxic, df_hate, df_sent], ignore_index=True)

print(f"Tổng số dòng văn bản thu thập từ 3 dataset: {len(df)}")

# Lọc bỏ giá trị rỗng và trùng lặp
df = df.dropna(subset=['text', 'label_raw'])

df = df.drop_duplicates(
    subset=['task', 'text', 'label_raw']
).reset_index(drop=True)
print(f"Số dòng sau khi lọc rỗng và trùng lặp: {len(df)}")

In [ ]:
# Cài đặt thư viện tách từ nếu chưa có:
!pip install pyvi

import re
from pyvi import ViTokenizer

# 1. Từ điển chuẩn hóa teencode tiếng Việt thông dụng
teencode_dict = {
    "ko": "không", "k": "không", "kh": "không", "hok": "không", "hong": "không",
    "dc": "được", "dk": "được", "đc": "được",
    "vs": "với", "j": "gì", "r": "rồi", "cx": "cũng",
    "thik": "thích", "thjx": "thích", "mik": "mình",
    "t": "tao", "m": "mày", "ae": "anh em",
    "ng": "người", "ntn": "như thế nào", "sao": "sao",
    "đt": "điện thoại", "fb": "facebook", "ib": "nhắn tin",
    "bt": "bình thường"
    # Lưu ý: "vl", "vcl", "cl" nên cân nhắc giữ nguyên nếu phân loại Hate Speech/Toxicity
}

def replace_teencode(text):
    words = text.split()
    return " ".join([teencode_dict.get(w, w) for w in words])

def process_emoji(text):
    # Nếu muốn dịch emoji sang text tiếng Anh, có thể dùng demojize:
    # return emoji.demojize(text, delimiters=(" ", " "))
    # Hoặc để nguyên emoji/chuẩn hóa tùy bài toán. Tạm thời giữ hàm nếu bạn cần:
    return emoji.demojize(text, delimiters=(" ", " "))

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # 1. Chuyển chữ thường
    text = text.lower()

    # 2. Xóa URL (sửa regex chuẩn để không khớp nhầm từ)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 3. Xóa mention (@username) và hashtag (#tag)
    text = re.sub(r'[@#]\w+', '', text)

    # 4. Rút gọn ký tự lặp quá 2 lần (vd: "ngonnnn" -> "ngon")
    text = re.sub(r'([a-zà-ỹ])\1{2,}', r'\1', text)

    # 5. Rút gọn dấu câu bị spam lặp lại (vd: "???" -> "?", "!!!!" -> "!")
    text = re.sub(r'([.?!])\1+', r'\1', text)

    # 6. Chuẩn hóa teencode
    text = replace_teencode(text)

    # 7. Xóa ký tự lạ/rác, NHƯNG GIỮ LẠI: chữ, số, tiếng Việt và các dấu . , ? !
    text = re.sub(r'[^0-9a-zà-ỹ\s.,?!]', ' ', text)

    # 8. Tách khoảng trắng quanh dấu câu để tránh dính liền từ
    text = re.sub(r'\s*([.,?!])\s*', r' \1 ', text)

    # 9. Chuẩn hóa khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()

    # 10. TÁCH TỪ TIẾNG VIỆT (Bắt buộc chuẩn theo PhoBERT)
    # Ví dụ: "rất tuyệt vời !" -> "rất tuyệt_vời !"
    text = ViTokenizer.tokenize(text)

    return text

In [ ]:
print("Đang tiến hành làm sạch văn bản...")
df['text_cleaned'] = df['text'].apply(clean_text)

# Lọc bỏ các dòng bị rỗng sau khi làm sạch
df = df[df['text_cleaned'].str.strip() != ""].reset_index(drop=True)

# Hiển thị đối chiếu 5 dòng đầu
print("\n--- ĐỐI CHIẾU TRƯỚC VÀ SAU LÀM SẠCH ---")
for i in range(min(5, len(df))):
    print(f"Gốc : {df.loc[i, 'text']}")
    print(f"Sạch: {df.loc[i, 'text_cleaned']}")
    print("-" * 50)

# Lưu checkpoint ra Google Drive để Người 2 phụ trách mục 4.2 dùng tiếp
output_path = "/content/drive/MyDrive/df_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"\n[Hoàn thành 4.1] Đã lưu dữ liệu tại: {output_path}")

# 4.2 Tách từ và tokenize

In [ ]:
# Cài đặt thư viện cần thiết
!pip install pyvi transformers matplotlib seaborn pandas -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyvi import ViTokenizer
from transformers import AutoTokenizer

In [ ]:
# Đọc file dữ liệu đã làm sạch từ Mục 4.1
input_path = "/content/drive/MyDrive/df_cleaned.csv"
try:
    df = pd.read_csv(input_path)
    print(f"Đã tải dữ liệu từ Google Drive: {input_path}")
except Exception as e:
    print("Không tìm thấy file trên Drive, thử đọc file local 'df_cleaned.csv'...")
    df = pd.read_csv("df_cleaned.csv")

# Đảm bảo các dòng rỗng (NaN) ở cột text_cleaned được xử lý
df['text_cleaned'] = df['text_cleaned'].fillna('')

# Tiến hành tách từ ghép tiếng Việt bằng pyvi
print("Đang tiến hành tách từ ghép tiếng Việt (ViTokenizer)...")
df['text_segmented'] = df['text_cleaned'].apply(lambda x: ViTokenizer.tokenize(str(x)))

# Lưu kết quả ra file df_segmented.csv
output_path = "/content/drive/MyDrive/df_segmented.csv"
df.to_csv(output_path, index=False)
df.to_csv("df_segmented.csv", index=False) # Bản sao local
print(f"Đã lưu file kết quả tại: {output_path}")

# Kiểm tra thử 3 dòng đầu
df[['task', 'text_cleaned', 'text_segmented']].head(3)

In [ ]:
# Khởi tạo PhoBERT Tokenizer
MODEL_NAME = "vinai/phobert-base-v2"
print(f"Đang tải tokenizer của PhoBERT ({MODEL_NAME})...")
# Không cần dùng token=True vì đây là mô hình public
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tính độ dài token của từng câu trong cột text_segmented
print("Đang tính toán độ dài token của toàn bộ tập dữ liệu...")
token_lengths = [len(tokenizer.encode(text, truncation=False)) for text in df['text_segmented']]

# Vẽ biểu đồ phân bố độ dài token
plt.figure(figsize=(10, 5))
sns.histplot(token_lengths, bins=50, kde=True, color='skyblue')
plt.title('Phân bố độ dài Token PhoBERT (Token Length Distribution)')
plt.xlabel('Số lượng Token')
plt.ylabel('Số lượng mẫu')
plt.grid(axis='y', alpha=0.75)
plt.show()

# Thống kê các mốc quan trọng
max_len_found = max(token_lengths)
p95 = int(np.percentile(token_lengths, 95))
p99 = int(np.percentile(token_lengths, 99))

print(f"--- THỐNG KÊ ĐỘ DÀI TOKEN ---")
print(f"- Tổng số mẫu: {len(token_lengths)}")
print(f"- Độ dài lớn nhất: {max_len_found} tokens")
print(f"- 95% câu có độ dài <= {p95} tokens")
print(f"- 99% câu có độ dài <= {p99} tokens")

In [ ]:
# Chốt tham số MAX_LEN (Bao phủ ~95-99% dữ liệu, không vượt quá 256 để tránh tràn RAM/GPU)
MAX_LEN = min(256, max(p95, 64))
print(f"CHỐT THAM SỐ MAX_LEN = {MAX_LEN}")

# Hàm tokenize chuẩn cho PhoBERT
def tokenize_func(texts, max_length=MAX_LEN):
    """
    Hàm tokenize nhận vào danh sách/Series văn bản đã tách từ (text_segmented)
    và trả về Dictionary chứa input_ids và attention_mask dạng PyTorch Tensor.
    """
    return tokenizer(
        list(texts),
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

# Test thử hàm tokenize_func trên 2 mẫu đầu tiên
sample_tokens = tokenize_func(df['text_segmented'].head(2))
print("\n[Kiểm tra chạy thử tokenize_func]:")
print("- Shape input_ids:", sample_tokens['input_ids'].shape)
print("- Shape attention_mask:", sample_tokens['attention_mask'].shape)
print("\nHOÀN THÀNH MỤC 4.2!")

# 4.3 Nhãn và đóng gói

 # 4. Train Toxicity model - Hoàng Minh


In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_from_disk
from sklearn.utils.class_weight import compute_class_weight

# 1. Thư mục lưu kết quả model và biểu đồ
SAVE_DIR = "/content/drive/MyDrive/toxicity_model_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# Khai báo model_name và tokenizer
model_name = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# 2. Lấy dữ liệu Toxicity đã đóng gói từ Mục 4.3
if 'hf_datasets' in globals() and 'toxicity' in hf_datasets:
    toxicity_dataset = hf_datasets['toxicity']
elif os.path.exists("/content/section_4_3/toxicity/hf_dataset"):
    toxicity_dataset = load_from_disk("/content/section_4_3/toxicity/hf_dataset")
else:
    # Dự phòng nếu nhóm lưu dữ liệu ở file df_segmented.csv
    from datasets import Dataset, DatasetDict
    df_all = pd.read_csv("/content/drive/MyDrive/df_segmented.csv")
    df_tox = df_all[df_all['task'] == 'toxicity'].copy().reset_index(drop=True)
    df_tox['label'] = df_tox['label_raw'].astype(int)

    from sklearn.model_selection import train_test_split
    tr, ts = train_test_split(df_tox, test_size=0.2, random_state=42, stratify=df_tox['label'])
    va, ts = train_test_split(ts, test_size=0.5, random_state=42, stratify=ts['label'])

    def tokenize_function(examples):
        return tokenizer(examples['text_segmented'], padding="max_length", truncation=True, max_length=128)

    ds_tr = Dataset.from_pandas(tr).map(tokenize_function, batched=True)
    ds_va = Dataset.from_pandas(va).map(tokenize_function, batched=True)
    ds_ts = Dataset.from_pandas(ts).map(tokenize_function, batched=True)
    toxicity_dataset = DatasetDict({'train': ds_tr, 'validation': ds_va, 'test': ds_ts})

print("✅ Đã chuẩn bị xong dữ liệu Toxicity:")
print(toxicity_dataset)

In [ ]:
# 3. Chuẩn bị Trọng số lớp (Class Weights) cho hàm Loss (0: non-toxic, 1: toxic)
if 'class_weights_dict' in globals() and 'toxicity' in class_weights_dict:
    weights_list = [
        class_weights_dict['toxicity'][0],
        class_weights_dict['toxicity'][1]
    ]
else:
    train_labels = np.array(toxicity_dataset['train']['label'] if 'label' in toxicity_dataset['train'].column_names else toxicity_dataset['train']['labels'])
    unique_classes = np.unique(train_labels)
    computed_weights = compute_class_weight('balanced', classes=unique_classes, y=train_labels)
    weights_list = computed_weights.tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = torch.tensor(weights_list, dtype=torch.float).to(device)
print("Device sử dụng:", device)
print("Trọng số Class Weights đưa vào Loss:", class_weights_tensor)

# 4. Kế thừa Trainer để tối ưu hàm Loss có trọng số (Macro F1 ưu tiên)
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 5. Hàm tính toán chỉ số đánh giá (Bắt buộc Macro F1)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )

    return {
        'accuracy': acc,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro
    }

In [ ]:
# 6. Nạp kiến trúc PhoBERT cho 2 nhãn phân loại (Toxicity)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "non-toxic", 1: "toxic"},
    label2id={"non-toxic": 0, "toxic": 1}
)

training_args = TrainingArguments(
    output_dir=f"{SAVE_DIR}/checkpoints",
    num_train_epochs=3,                     # Huấn luyện chuẩn 3 epoch
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",                  # Đánh giá sau mỗi epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",       # Tự động chọn epoch có Macro F1 cao nhất
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=toxicity_dataset['train'],
    eval_dataset=toxicity_dataset['validation'],
    compute_metrics=compute_metrics
)

print("🚀 Bắt đầu huấn luyện mô hình Toxicity PhoBERT...")
trainer.train()

In [ ]:
print("Tiến hành đánh giá trên tập độc lập Test...")
test_predictions = trainer.predict(toxicity_dataset['test'])

# Lấy nhãn thực tế
if 'labels' in toxicity_dataset['test'].column_names:
    y_true = np.array(toxicity_dataset['test']['labels'])
else:
    y_true = np.array(toxicity_dataset['test']['label'])

y_pred = np.argmax(test_predictions.predictions, axis=-1)
target_names = ["non-toxic", "toxic"]

# 1. Tính toán các chỉ số cơ bản
acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
cls_report_dict = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)
cls_report_str = classification_report(y_true, y_pred, target_names=target_names)
cm = confusion_matrix(y_true, y_pred)

# 2. In trực quan ra màn hình
print("=" * 60)
print(f"Accuracy        : {acc:.4f}")
print(f"Precision Macro : {prec:.4f}")
print(f"Recall Macro    : {rec:.4f}")
print(f"F1-score Macro  : {f1:.4f} (Chỉ số ưu tiên)")
print("=" * 60)
print("\n--- CLASSIFICATION REPORT ---")
print(cls_report_str)

# 3. Vẽ và lưu Confusion Matrix ra file ảnh
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title("Confusion Matrix - Toxicity Analysis")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
cm_path = f"{SAVE_DIR}/confusion_matrix_toxicity.png"
plt.savefig(cm_path, dpi=300)
plt.show()

# 4. Lưu toàn bộ chỉ số vào file CSV trên Google Drive
metrics_df = pd.DataFrame(cls_report_dict).transpose()
metrics_df.to_csv(f"{SAVE_DIR}/toxicity_classification_report.csv")

# 5. Lưu lại Model và Tokenizer hoàn thiện
model.save_pretrained(f"{SAVE_DIR}/final_model")
tokenizer.save_pretrained(f"{SAVE_DIR}/final_model")
print(f"\n[Hoàn thành] Toàn bộ Model, Chỉ số và Biểu đồ đã lưu tại:\n-> {SAVE_DIR}")

#5. Train Hate Speech model - Tú

In [ ]:
# ============================================================
# 5. TRAIN HATE SPEECH MODEL - BƯỚC 1: SETUP
# ============================================================

import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_from_disk
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)

# Để kết quả có thể tái lập
set_seed(42)

# ------------------------------------------------------------
# Kiểm tra GPU
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Chưa bật GPU!")

# ------------------------------------------------------------
# Cấu hình
# ------------------------------------------------------------

SAVE_DIR = "/content/drive/MyDrive/hate_speech_model_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# Team đang dùng PhoBERT base-v2 ở Toxicity và Sentiment
model_name = "vinai/phobert-base-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("\nModel:", model_name)
print("Save directory:", SAVE_DIR)

In [ ]:
# ============================================================
# RETOKENIZE HATE SPEECH BẰNG PHOBERT-BASE-V2
# ============================================================

import pandas as pd

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

MODEL_NAME = "vinai/phobert-base-v2"
MAX_LEN = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_dir = "/content/section_4_3/hate_speech"

# Đọc 3 tập đã chia từ mục 4.3
train_df = pd.read_csv(f"{base_dir}/train.csv")
val_df   = pd.read_csv(f"{base_dir}/validation.csv")
test_df  = pd.read_csv(f"{base_dir}/test.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nColumns:")
print(train_df.columns.tolist())

# Chỉ giữ cột cần thiết
def make_dataset(df):
    temp = df[["text_segmented", "labels"]].copy()
    temp["labels"] = temp["labels"].astype("int64")

    return Dataset.from_pandas(
        temp,
        preserve_index=False
    )

hate_dataset_v2 = DatasetDict({
    "train": make_dataset(train_df),
    "validation": make_dataset(val_df),
    "test": make_dataset(test_df)
})

def tokenize_batch(batch):
    return tokenizer(
        batch["text_segmented"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

hate_dataset_v2 = hate_dataset_v2.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text_segmented"]
)

hate_dataset_v2.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

# Lưu phiên bản mới
save_path = (
    "/content/section_4_3/"
    "hate_speech/hf_dataset_phobert_v2"
)

hate_dataset_v2.save_to_disk(save_path)

print("\nDONE!")
print(hate_dataset_v2)
print("Đã lưu:", save_path)

sample = hate_dataset_v2["train"][0]

print("\nSample:")
print("input_ids:", sample["input_ids"].shape)
print("attention_mask:", sample["attention_mask"].shape)
print("label:", sample["labels"].item())

In [ ]:
# ============================================================
# HATE SPEECH - CLASS WEIGHTS
# ============================================================

import numpy as np
import torch

from sklearn.utils.class_weight import compute_class_weight

# Lấy label của TRAIN SET
y_train = np.array(
    hate_dataset_v2["train"]["labels"]
)

classes = np.array([0, 1, 2])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights_tensor = torch.tensor(
    weights,
    dtype=torch.float
).to(device)

print("Số mẫu từng lớp:")

unique, counts = np.unique(
    y_train,
    return_counts=True
)

for label, count in zip(unique, counts):
    print(f"Class {label}: {count}")

print("\nClass Weights:")

for label, weight in zip(classes, weights):
    print(f"Class {label}: {weight:.4f}")

print("\nTensor:")
print(class_weights_tensor)

In [ ]:
import torch.nn as nn

from transformers import Trainer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights_tensor
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(
        labels,
        preds
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="macro",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

print("WeightedTrainer ready!")

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base-v2",
    num_labels=3,
    id2label={
        0: "CLEAN",
        1: "OFFENSIVE",
        2: "HATE"
    },
    label2id={
        "CLEAN": 0,
        "OFFENSIVE": 1,
        "HATE": 2
    }
)

print("Model loaded!")
print(model.config.id2label)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/hate_speech_model_results/checkpoints",

    num_train_epochs=3,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_steps=50,

    fp16=True,

    report_to="none",

    seed=42
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,

    train_dataset=hate_dataset_v2["train"],
    eval_dataset=hate_dataset_v2["validation"],

    compute_metrics=compute_metrics
)

print("Trainer ready!")

In [ ]:
import os

SAVE_DIR = "/content/drive/MyDrive/hate_speech_model_results"
FINAL_PATH = os.path.join(SAVE_DIR, "final_model")

os.makedirs(FINAL_PATH, exist_ok=True)

# Lưu tokenizer đang sử dụng trong phần huấn luyện Hate Speech
tokenizer.save_pretrained(FINAL_PATH)

print("Đường dẫn:", FINAL_PATH)
print("Các file đã lưu:")
for file in os.listdir(FINAL_PATH):
    print("-", file)

#6. Train Sentiment model - Hưng

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_from_disk

# 1. Thư mục lưu kết quả model và biểu đồ
SAVE_DIR = "/content/drive/MyDrive/sentiment_model_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# 2. Lấy dữ liệu Sentiment đã đóng gói từ Mục 4.3
if 'hf_datasets' in globals() and 'sentiment' in hf_datasets:
    sentiment_dataset = hf_datasets['sentiment']
else:
    sentiment_dataset = load_from_disk("/content/section_4_3/sentiment/hf_dataset")

# 3. Chuẩn bị Trọng số lớp (Class Weights) cho hàm Loss
# Sentiment gồm 3 lớp: 0: negative, 1: neutral, 2: positive
weights_list = [
    class_weights_dict['sentiment'][0],
    class_weights_dict['sentiment'][1],
    class_weights_dict['sentiment'][2]
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = torch.tensor(weights_list, dtype=torch.float).to(device)
print("Device sử dụng:", device)
print("Trọng số Class Weights đưa vào Loss:", class_weights_tensor)

# 4. Kế thừa Trainer để tối ưu hàm Loss có trọng số (Macro F1 ưu tiên)
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 5. Hàm tính toán chỉ số đánh giá (Bắt buộc Macro F1)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )

    return {
        'accuracy': acc,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro
    }

# 6. Nạp kiến trúc PhoBERT cho 3 nhãn phân loại
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label={0: "negative", 1: "neutral", 2: "positive"},
    label2id={"negative": 0, "neutral": 1, "positive": 2}
)

In [ ]:
training_args = TrainingArguments(
    output_dir=f"{SAVE_DIR}/checkpoints",
    num_train_epochs=3,                     # Huấn luyện chuẩn từ 3-4 epoch
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",                  # Đánh giá sau mỗi epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",       # Tự động chọn epoch có Macro F1 cao nhất
    greater_is_better=True,
    logging_steps=50,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=sentiment_dataset['train'],
    eval_dataset=sentiment_dataset['validation'],
    compute_metrics=compute_metrics
)

print("Bắt đầu huấn luyện mô hình Sentiment PhoBERT...")
trainer.train()

In [ ]:
print("Tiến hành đánh giá trên tập độc lập Test...")
test_predictions = trainer.predict(sentiment_dataset['test'])
y_true = np.array(sentiment_dataset['test']['labels'])
y_pred = np.argmax(test_predictions.predictions, axis=-1)
target_names = ["negative", "neutral", "positive"]

# 1. Tính toán các chỉ số cơ bản
acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
cls_report_dict = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)
cls_report_str = classification_report(y_true, y_pred, target_names=target_names)
cm = confusion_matrix(y_true, y_pred)

# 2. In trực quan ra màn hình
print("=" * 60)
print(f"Accuracy        : {acc:.4f}")
print(f"Precision Macro : {prec:.4f}")
print(f"Recall Macro    : {rec:.4f}")
print(f"F1-score Macro  : {f1:.4f} (Chỉ số ưu tiên)")
print("=" * 60)
print("\n--- CLASSIFICATION REPORT ---")
print(cls_report_str)

# 3. Vẽ và lưu Confusion Matrix ra file ảnh
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title("Confusion Matrix - Sentiment Analysis")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
cm_path = f"{SAVE_DIR}/confusion_matrix_sentiment.png"
plt.savefig(cm_path, dpi=300)
plt.show()

# 4. Lưu toàn bộ chỉ số vào file Excel / CSV trên Google Drive
metrics_df = pd.DataFrame(cls_report_dict).transpose()
metrics_df.to_csv(f"{SAVE_DIR}/sentiment_classification_report.csv")

# 5. Lưu lại Model và Tokenizer hoàn thiện
model.save_pretrained(f"{SAVE_DIR}/final_model")
tokenizer.save_pretrained(f"{SAVE_DIR}/final_model")
print(f"\n[Hoàn thành 6] Toàn bộ Model, Chỉ số và Biểu đồ đã lưu tại:\n-> {SAVE_DIR}")

#7. Đánh giá 3 model - Bình Minh

#8. Phần thử nghiệm - Bình Minh


#9. Đánh giá thông số - Vương

#10. Lưu model - Vương
